# NextGen-HR — Python Backend (Google Colab)

### How to use:
1. Run **Cell 1** — installs packages
2. Run **Cell 2** — loads all ML algorithms
3. Run **Cell 3** — registers Flask API routes
4. Run **Cell 4** — starts server and prints your ngrok URL
5. Copy the ngrok URL and paste it into the React app Backend URL field

Keep this Colab tab open while using the app. Closing it stops the backend.

In [ ]:
# CELL 1 - Install required packages
!pip install flask flask-cors pyngrok pymupdf -q
print('All packages installed successfully')

In [ ]:
# CELL 2 - All Algorithm Implementations

import math
import re
import random
import base64
import copy
from collections import Counter

STOP = set([
    'a','an','the','and','or','but','in','on','at','to','for','of','with','by',
    'from','is','are','was','were','be','been','have','has','had','do','does',
    'did','will','would','could','should','not','it','its','this','that','i',
    'you','he','she','we','they','my','your','how','what','when','where','why',
    'all','so','as','if','up','out','about','into','can','may','must','than',
    'then','there','their','which','who','also','just','very','more','some',
    'such','each','after','before','over','same','being','between','through',
    'during','without','under','while','these','those','them','our','its','use',
])

def tok(text):
    words = re.sub(r'[^a-z0-9\s\-]', ' ', text.lower()).split()
    return [w for w in words if len(w) > 2 and w not in STOP]

def tfidf_fit(docs):
    df = {}
    n = len(docs)
    for d in docs:
        for t in set(tok(d)):
            df[t] = df.get(t, 0) + 1
    vocab, idf = {}, {}
    for i, t in enumerate(df.keys()):
        vocab[t] = i
        idf[t] = math.log((n + 1) / (df[t] + 1)) + 1
    return vocab, idf, len(vocab)

def tfidf_vec(text, vocab, idf, dim):
    tokens = tok(text)
    if not tokens:
        return [0.0] * max(dim, 1)
    tf = Counter(tokens)
    v = [0.0] * max(dim, 1)
    for t, c in tf.items():
        if t in vocab:
            v[vocab[t]] = (c / len(tokens)) * idf.get(t, 1)
    return v

def cosine(a, b):
    n = min(len(a), len(b))
    d  = sum(a[i] * b[i] for i in range(n))
    na = math.sqrt(sum(x * x for x in a))
    nb = math.sqrt(sum(x * x for x in b))
    return d / (na * nb) if na and nb else 0.0

def extract_topics(job_description, k=10):
    vocab, idf, dim = tfidf_fit([job_description])
    tokens   = tok(job_description)
    bigrams  = [tokens[i]+' '+tokens[i+1] for i in range(len(tokens)-1)]
    trigrams = [tokens[i]+' '+tokens[i+1]+' '+tokens[i+2] for i in range(len(tokens)-2)]
    candidates = list(set(tokens + bigrams + trigrams))
    def score_term(term):
        tt = tok(term)
        if not tt: return 0
        s = sum(idf.get(t, 0) for t in tt) / len(tt)
        s *= (1 + len(tt) * 0.3)
        if len(tt) == 1 and len(tt[0]) < 5: s *= 0.6
        return s
    scored = sorted([(term, score_term(term)) for term in candidates], key=lambda x: -x[1])
    return [t for t, _ in scored[:k]]

FRAMES = [
    lambda t,t2: f'Tell me about a time you worked directly with {t}. What was your role and what did you accomplish?',
    lambda t,t2: f'Describe your most significant experience involving {t}. What were the key decisions you made?',
    lambda t,t2: f'Walk me through how you have applied {t} in a real professional context.',
    lambda t,t2: f'How do you approach {t} when starting from scratch? Walk me through your process step by step.',
    lambda t,t2: f'What factors do you consider when making decisions about {t}?',
    lambda t,t2: f'How do you evaluate whether your approach to {t} is working? What signals do you watch for?',
    lambda t,t2: f'Describe a challenging situation you faced involving {t}. How did you diagnose and resolve it?',
    lambda t,t2: f'What can go wrong with {t}, and how have you handled those situations?',
    lambda t,t2: f'Tell me about a time when your approach to {t} did not work as expected. What did you learn?',
    lambda t,t2: f'Compare two different approaches you have used for {t}. What were the trade-offs?',
    lambda t,t2: f'How does {t} interact with {t2} in practice? Describe a situation where both mattered.',
    lambda t,t2: f'You need to balance {t} against {t2} under tight constraints. How do you decide?',
    lambda t,t2: f'What is the most impactful outcome you achieved through your work on {t}?',
    lambda t,t2: f'How have you improved or optimised {t} in a previous role? Be specific about what changed and by how much.',
    lambda t,t2: f'If a colleague unfamiliar with {t} asked for your help, how would you guide them and how does that connect to {t2}?',
    lambda t,t2: f'What does excellent work look like in the context of {t}? How do you know when you have achieved it?',
    lambda t,t2: f'Describe a project where {t} and {t2} were both critical. How did you manage the interplay?',
    lambda t,t2: f'How has your understanding of {t} evolved over your career? What changed your thinking?',
]

def generate_questions(job_description, existing_qids=None):
    existing_qids = set(existing_qids or [])
    topics = extract_topics(job_description, 12)
    if not topics: return []
    questions, used = [], set()
    for ti, topic in enumerate(topics):
        topic2 = topics[(ti + 3) % len(topics)]
        for fi, frame in enumerate(FRAMES):
            key = f'{ti}_{fi}'
            if key in used: continue
            qid = f'q_{ti}_{fi}'
            if qid not in existing_qids:
                questions.append({'id': qid, 'text': frame(topic, topic2),
                                  'topic': topic, 'topic2': topic2, 'fi': fi})
                used.add(key)
    return questions

def sigma(x):
    x = max(-20.0, min(20.0, float(x)))
    return 1.0 / (1.0 + math.exp(-x))

def irt_prob(item, theta):
    return sigma(item['a'] * (theta - item['b']))

def irt_info(item, theta):
    p = irt_prob(item, theta)
    return item['a'] ** 2 * p * (1 - p)

def irt_update(items, qid, score, theta):
    items = copy.deepcopy(items)
    if qid not in items:
        items[qid] = {'b': 0.0, 'a': 1.0, 'n': 0}
    item = items[qid]
    item['n'] += 1
    lr  = 0.20 / math.sqrt(item['n'])
    p   = irt_prob(item, theta)
    err = score - p
    item['b'] = max(-4.0, min(4.0, item['b'] + lr * (-item['a'] * err)))
    item['a'] = max(0.2,  min(4.0, item['a'] + lr * ((theta - item['b']) * err)))
    return items

def estimate_theta(items, responses):
    if not responses: return 0.0
    theta = 0.0
    for _ in range(25):
        g = H = 0.0
        for r in responses:
            qid, score = r['qid'], r['score']
            if qid not in items: continue
            item = items[qid]
            p  = irt_prob(item, theta)
            g += item['a'] * (score - p)
            H -= item['a'] ** 2 * p * (1 - p)
        if abs(H) < 1e-8: break
        theta -= max(-0.5, min(0.5, g / H))
    return max(-4.0, min(4.0, theta))

def beta_sample(a, b):
    for _ in range(1000):
        x = random.random() ** (1.0 / a)
        y = random.random() ** (1.0 / b)
        if 0 < x + y <= 1:
            return x / (x + y)
    return 0.5

def bandit_select(arms, available_qids, irt_items, theta):
    if not available_qids: return None
    best_qid, best_score = available_qids[0], float('-inf')
    for qid in available_qids:
        if qid not in arms:
            arms[qid] = {'alpha': 1.0, 'beta': 1.0, 'n': 0}
        arm  = arms[qid]
        ts   = beta_sample(arm['alpha'], arm['beta'])
        info = irt_info(irt_items[qid], theta) if (qid in irt_items and irt_items[qid]['n'] > 0) else 0.5
        s    = ts * 0.6 + info * 0.4
        if s > best_score:
            best_score, best_qid = s, qid
    return best_qid

def bandit_update(arms, qid, reward):
    arms = copy.deepcopy(arms)
    if qid not in arms:
        arms[qid] = {'alpha': 1.0, 'beta': 1.0, 'n': 0}
    arms[qid]['n']     += 1
    arms[qid]['alpha'] += reward
    arms[qid]['beta']  += (1 - reward)
    return arms

def score_answer(question, answer, job_desc):
    text  = (answer or '').strip()
    words = [w for w in text.split() if w]
    wc    = len(words)
    if wc < 4:
        return {'score': 0.04, 'signals': {}, 'tip': 'Please give a more complete answer.',
                'wc': wc, 'relevance': 0, 'specificityScore': 0, 'richness': 0}
    context = question + ' ' + job_desc[:500]
    vocab, idf_map, dim = tfidf_fit([text, context])
    relevance = cosine(tfidf_vec(text, vocab, idf_map, dim), tfidf_vec(context, vocab, idf_map, dim))
    lo = text.lower()
    signals = {
        'hasAction':  bool(re.search(r'\b(implemented|built|designed|developed|created|led|managed|launched|delivered|introduced|established|improved|reduced|increased|solved|resolved|negotiated|prepared|executed|coordinated|mentored|trained|deployed|migrated|optimised|produced|directed|founded|grew|saved|automated|restructured)\b', text, re.I)),
        'hasOutcome': bool(re.search(r'\b(result|outcome|impact|achieved|improved|increased|reduced|saved|delivered|enabled|successfully|led to|as a result|consequently|percent|%|doubled|tripled|halved)\b', lo)),
        'hasNumbers': bool(re.search(r'\d+', text)),
        'hasContext': bool(re.search(r'\b(when|during|while|at my|in my|our team|the project|the client|at that point|at the time)\b', lo)),
        'goodLength': 40 <= wc <= 280,
        'notVague':   not bool(re.search(r'\b(just|basically|kind of|sort of|maybe|probably|i think|i guess|i believe|not sure|generally)\b', lo)),
    }
    spec_score = sum(signals.values()) / len(signals)
    tokens  = tok(text)
    richness = len(set(tokens)) / len(tokens) if tokens else 0
    raw   = relevance * 0.40 + spec_score * 0.42 + richness * 0.18
    score = min(max(raw, 0), 1.0)
    if not signals['hasAction']:    tip = 'Use action verbs - describe what YOU did (built, led, resolved, designed)'
    elif not signals['hasOutcome']: tip = 'State the outcome or result of your action'
    elif not signals['hasNumbers']: tip = 'Add a number, percentage, or scale to make your answer concrete'
    elif not signals['goodLength']: tip = 'Expand your answer - aim for at least 40 words' if wc < 40 else 'Your answer is very long - try to be more concise'
    elif not signals['hasContext']: tip = 'Set context: when/where did this happen?'
    else: tip = 'Strong, specific answer.' if score > 0.72 else 'Good - add more concrete evidence.'
    return {'score': score, 'relevance': relevance, 'specificityScore': spec_score,
            'richness': richness, 'signals': signals, 'tip': tip, 'wc': wc}

def score_resume_func(resume, job_desc):
    vocab, idf_map, dim = tfidf_fit([resume, job_desc])
    sim = cosine(tfidf_vec(resume, vocab, idf_map, dim), tfidf_vec(job_desc, vocab, idf_map, dim))
    jd_terms = extract_topics(job_desc, 15)
    rlo      = resume.lower()
    covered  = [t for t in jd_terms if t in rlo]
    coverage = len(covered) / len(jd_terms) if jd_terms else 0
    exp_m    = re.search(r'(\d+)\+?\s*years?\b', resume, re.I)
    exp_yrs  = int(exp_m.group(1)) if exp_m else 0
    exp_score = min(exp_yrs / 10.0, 1.0)
    final = sim * 0.45 + coverage * 0.40 + exp_score * 0.15
    return {'score': min(final, 1.0), 'sim': sim, 'coverage': coverage,
            'expScore': exp_score, 'expYrs': exp_yrs, 'jdTerms': jd_terms,
            'covered': covered, 'missing': [t for t in jd_terms if t not in rlo]}

def final_score_func(qas, irt_items):
    if not qas:
        return {'score': 0, 'theta': 0, 'slope': 0, 'progression': [], 'thetaNorm': 0, 'infoWt': 0, 'raw': []}
    responses  = [{'qid': qa['qid'], 'score': qa['score']} for qa in qas]
    theta      = estimate_theta(irt_items, responses)
    theta_norm = min(max((theta + 4) / 8.0, 0), 1)
    info_w = [max(irt_info(irt_items[qa['qid']], theta) if qa['qid'] in irt_items else 0.05, 0.05) for qa in qas]
    sum_w  = sum(info_w)
    info_wt = sum(qa['score'] * info_w[i] for i, qa in enumerate(qas)) / sum_w
    scores = [qa['score'] for qa in qas]
    n  = len(scores)
    mx = (n - 1) / 2.0
    my = sum(scores) / n
    denom = sum((i - mx) ** 2 for i in range(n)) or 0.001
    slope = sum((i - mx) * (s - my) for i, s in enumerate(scores)) / denom
    progression = []
    for i in range(n):
        w = scores[max(0, i-2):i+1]
        progression.append(sum(w) / len(w))
    blended = theta_norm * 0.55 + info_wt * 0.45
    return {'score': min(max(blended, 0), 1), 'theta': theta, 'thetaNorm': theta_norm,
            'infoWt': info_wt, 'slope': slope, 'progression': progression, 'raw': scores}

print('Cell 2 complete - all algorithms loaded')

In [ ]:
# CELL 3 - Flask API Routes

from flask import Flask, request, jsonify
from flask_cors import CORS
import fitz

app = Flask(__name__)
CORS(app)

@app.route('/api/health', methods=['GET'])
def health():
    return jsonify({'status': 'ok', 'message': 'NextGen-HR backend is running'})

@app.route('/api/score-resume', methods=['POST'])
def api_score_resume():
    data = request.get_json()
    return jsonify(score_resume_func(data['resume'], data['jobDescription']))

@app.route('/api/generate-questions', methods=['POST'])
def api_generate_questions():
    data = request.get_json()
    return jsonify({'questions': generate_questions(data['jobDescription'], data.get('existingQids', []))})

@app.route('/api/score-answer', methods=['POST'])
def api_score_answer():
    data = request.get_json()
    return jsonify(score_answer(data['question'], data['answer'], data['jobDescription']))

@app.route('/api/irt-update', methods=['POST'])
def api_irt_update():
    data = request.get_json()
    return jsonify({'irtItems': irt_update(data['irtItems'], data['qid'], data['score'], data['theta'])})

@app.route('/api/estimate-theta', methods=['POST'])
def api_estimate_theta():
    data = request.get_json()
    return jsonify({'theta': estimate_theta(data['irtItems'], data['responses'])})

@app.route('/api/bandit-select', methods=['POST'])
def api_bandit_select():
    data = request.get_json()
    return jsonify({'selectedQid': bandit_select(data['banditArms'], data['availableQids'], data['irtItems'], data['theta'])})

@app.route('/api/bandit-update', methods=['POST'])
def api_bandit_update():
    data = request.get_json()
    return jsonify({'banditArms': bandit_update(data['banditArms'], data['qid'], data['reward'])})

@app.route('/api/final-score', methods=['POST'])
def api_final_score():
    data = request.get_json()
    return jsonify(final_score_func(data['qas'], data['irtItems']))

@app.route('/api/extract-pdf', methods=['POST'])
def api_extract_pdf():
    data = request.get_json()
    pdf_bytes = base64.b64decode(data['base64Data'])
    doc  = fitz.open(stream=pdf_bytes, filetype='pdf')
    text = ''.join(page.get_text() + '\n' for page in doc)
    doc.close()
    if not text.strip():
        return jsonify({'error': 'Could not extract text from PDF'}), 400
    return jsonify({'text': text.strip()})

print('Cell 3 complete - Flask routes registered')

In [ ]:
# CELL 4 - Start Server + ngrok
#
# BEFORE RUNNING:
#   1. Go to https://dashboard.ngrok.com/ and sign up free
#   2. Copy your Auth Token from the dashboard
#   3. Replace YOUR_NGROK_TOKEN_HERE below with your actual token

import threading
from pyngrok import ngrok

# PASTE YOUR NGROK TOKEN HERE
NGROK_TOKEN = 'YOUR_NGROK_TOKEN_HERE'

ngrok.set_auth_token(NGROK_TOKEN)
ngrok.kill()

public_url = ngrok.connect(5000)

print('=' * 60)
print('NextGen-HR Backend is LIVE!')
print('=' * 60)
print(f'\nYour Backend URL:\n\n    {public_url}\n')
print('Copy the URL above and paste it into')
print('the Backend URL field in your React app.')
print('=' * 60)
print('\nKeep this cell running. Do not close Colab.\n')

def run_server():
    app.run(host='0.0.0.0', port=5000, use_reloader=False, debug=False)

server_thread = threading.Thread(target=run_server)
server_thread.daemon = True
server_thread.start()